# Evaluación contra Ground Truth

## Resultados finales

| Métrica | Valor |
|---|---|
| **Precision** | 87.5% |
| **Recall** | 82.4% |
| **F1** | 84.8% |
| **Imágenes perfectas** | 12/17 |
| TP | 28 |
| FP | 4 |
| FN | 6 |

## Mejoras aplicadas respecto a versión inicial (F1=63%)

| Fix | Impacto |
|---|---|
| `detect_seps_vert_v2`: ventana fija en lugar de expansión libre | +14% F1 |
| Filtro `min_peak_brightness=150` en strips verticales | Elimina FPs en bordes y pliegues |
| `edge_margin=0.10` en rubber extent para strips verticales | Evita detección de bordes superior/inferior |
| `strip_edge_margin=0.06` en detección horizontal | Elimina FPs en bordes de strip |
| Column-profile fallback en detect_seps_horiz | Recupera gaps parciales que CC no captura |

In [1]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import os
import json
from scipy.signal import find_peaks

DATASET_PATH  = "../../data/michelin_dataset"
GT_PATH       = "ground_truth.json"  # generado por extract_red_bboxes
MATCH_DIST_PX = 150  # distancia máxima de centros para considerar match

## Extracción de ground truth desde anotaciones rojas

In [2]:
def extract_red_bboxes(img_path):
    """
    Detecta los bounding boxes rojos dibujados manualmente en la imagen.
    Devuelve lista de dicts {x, y, w, h, cx, cy}.
    """
    img = cv2.imread(img_path)
    H, W = img.shape[:2]
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    mask1 = cv2.inRange(hsv, (0,   100, 80), (10,  255, 255))
    mask2 = cv2.inRange(hsv, (165, 100, 80), (180, 255, 255))
    red_mask = cv2.bitwise_or(mask1, mask2)
    k = np.ones((3,3), np.uint8)
    red_clean = cv2.morphologyEx(red_mask, cv2.MORPH_OPEN, k, iterations=1)
    red_clean = cv2.morphologyEx(red_clean, cv2.MORPH_CLOSE, k, iterations=2)
    cnts, _ = cv2.findContours(red_clean, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    bboxes = []
    for cnt in cnts:
        if cv2.contourArea(cnt) < 200: continue
        x, y, w, h = cv2.boundingRect(cnt)
        bboxes.append({'x':int(x),'y':int(y),'w':int(w),'h':int(h),'cx':int(x+w//2),'cy':int(y+h//2)})
    bboxes.sort(key=lambda b: b['y']*10000+b['x'])
    return bboxes, H, W


def build_ground_truth(dataset_path, save_path=None):
    """Extrae GT de todas las imágenes anotadas y lo guarda en JSON."""
    files = sorted([f for f in os.listdir(dataset_path) if f.endswith('.jpg')])
    gt = {}
    for fname in files:
        bboxes, H, W = extract_red_bboxes(os.path.join(dataset_path, fname))
        gt[fname] = {'image_size': (W, H), 'n_separations': len(bboxes), 'bboxes': bboxes}
    if save_path:
        with open(save_path, 'w') as f:
            json.dump(gt, f, indent=2)
    return gt


# Cargar o construir GT
if os.path.exists(GT_PATH):
    with open(GT_PATH) as f:
        gt = json.load(f)
    print(f"GT cargado: {len(gt)} imágenes")
else:
    gt = build_ground_truth(DATASET_PATH, save_path=GT_PATH)
    print(f"GT generado y guardado: {len(gt)} imágenes")

GT cargado: 17 imágenes


## Pipeline completo (con todas las correcciones)

In [3]:
def segment_rubber_strips(gray, dark_thresh=80, min_area=50000):
    _, dark = cv2.threshold(gray, dark_thresh, 255, cv2.THRESH_BINARY_INV)
    k1, k2 = np.ones((25,25), np.uint8), np.ones((10,10), np.uint8)
    closed = cv2.morphologyEx(dark, cv2.MORPH_CLOSE, k1, iterations=3)
    opened = cv2.morphologyEx(closed, cv2.MORPH_OPEN, k2, iterations=2)
    cnts, _ = cv2.findContours(opened, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    strips = []
    for cnt in cnts:
        if cv2.contourArea(cnt) < min_area: continue
        x, y, w, h = cv2.boundingRect(cnt)
        strips.append({'bbox':(x,y,w,h), 'orientation':'horizontal' if w>=h else 'vertical'})
    strips.sort(key=lambda s: s['bbox'][1]*10000+s['bbox'][0])
    return strips


def multiscale_esm(gray_roi, scales=[1.0, 0.5, 0.25]):
    H_r, W_r = gray_roi.shape
    esm = np.zeros((H_r, W_r), dtype=np.float32)
    edm_x = np.zeros((H_r, W_r), dtype=np.float32)
    edm_y = np.zeros((H_r, W_r), dtype=np.float32)
    for scale in scales:
        sw, sh = max(1,int(W_r*scale)), max(1,int(H_r*scale))
        small = cv2.resize(gray_roi,(sw,sh),interpolation=cv2.INTER_AREA) if scale!=1.0 else gray_roi.copy()
        blur = cv2.GaussianBlur(small,(5,5),1.0)
        clahe = cv2.createCLAHE(2.0,(4,4)); enh = clahe.apply(blur)
        gx = cv2.Sobel(enh,cv2.CV_32F,1,0,ksize=3)
        gy = cv2.Sobel(enh,cv2.CV_32F,0,1,ksize=3)
        mag = np.sqrt(gx**2+gy**2)
        if scale!=1.0:
            mag=cv2.resize(mag,(W_r,H_r),interpolation=cv2.INTER_LINEAR)
            gx=cv2.resize(gx,(W_r,H_r),interpolation=cv2.INTER_LINEAR)
            gy=cv2.resize(gy,(W_r,H_r),interpolation=cv2.INTER_LINEAR)
        esm+=mag; edm_x+=gx; edm_y+=gy
    n = len(scales)
    return esm/n, edm_x/n, edm_y/n


def detect_seps_horiz(gray_roi, esm, edm_x, edm_y,
                       bright_gap_thresh=130, overlap_bright_min=90,
                       min_vert_span=0.20, max_sep_width=100,
                       context_pad=50, min_rubber_dark=0.20,
                       gap_merge_dist=60, strip_edge_margin=0.06):
    H_r, W_r = gray_roi.shape
    edge_px = int(W_r * strip_edge_margin)
    separations = []

    # A: Bright-gap connected components
    _, bright_bin = cv2.threshold(gray_roi, bright_gap_thresh, 255, cv2.THRESH_BINARY)
    k_v = np.ones((20,1), np.uint8)
    bright_closed = cv2.morphologyEx(bright_bin, cv2.MORPH_CLOSE, k_v, iterations=4)
    n_lbl, labels, stats, _ = cv2.connectedComponentsWithStats(bright_closed)
    for lbl in range(1, n_lbl):
        x=stats[lbl,cv2.CC_STAT_LEFT]; w=stats[lbl,cv2.CC_STAT_WIDTH]; h=stats[lbl,cv2.CC_STAT_HEIGHT]
        if h/H_r<min_vert_span or w>max_sep_width: continue
        if x<edge_px or x+w>W_r-edge_px: continue
        lz=gray_roi[:,max(0,x-context_pad):x]; rz=gray_roi[:,x+w:min(W_r,x+w+context_pad)]
        if lz.size==0 or rz.size==0: continue
        ld=np.sum(lz<80)/(lz.size+1e-6); rd=np.sum(rz<80)/(rz.size+1e-6)
        if ld<min_rubber_dark or rd<min_rubber_dark: continue
        mask=(labels==lbl); true_cont=float(np.sum(mask.any(axis=1))/H_r)
        separations.append({'x_center':int(x+w//2),'x_left':int(x),'x_right':int(x+w),
                             'width_px':int(w),'true_continuity':true_cont,'sep_type':'gap',
                             'mean_bright':float(gray_roi[mask].mean()) if mask.any() else 0,
                             'left_dark':float(ld),'right_dark':float(rd)})

    # B: Column-profile fallback
    col_mean = gray_roi.mean(axis=0).astype(np.float32)
    col_sm = cv2.GaussianBlur(col_mean.reshape(1,-1),(1,21),4).flatten()
    col_peaks, _ = find_peaks(col_sm, height=100, distance=25, prominence=20)
    for px in col_peaks:
        if px<edge_px or px>W_r-edge_px: continue
        if any(abs(px-s['x_center'])<gap_merge_dist for s in separations): continue
        col_bright = float(col_sm[px])
        if col_bright < overlap_bright_min: continue
        col_slice = gray_roi[:,max(0,px-5):min(W_r,px+6)]
        cont = float(np.sum(col_slice.mean(axis=1)>overlap_bright_min*0.7)/H_r)
        if cont < min_vert_span*0.5: continue
        xl, xr = px, px
        while xl>edge_px and col_sm[xl-1]>col_bright*0.6: xl-=1
        while xr<W_r-edge_px-1 and col_sm[xr+1]>col_bright*0.6: xr+=1
        if xr-xl>max_sep_width: continue
        lz=gray_roi[:,max(0,xl-context_pad):xl]; rz=gray_roi[:,xr:min(W_r,xr+context_pad)]
        if lz.size==0 or rz.size==0: continue
        ld=np.sum(lz<80)/(lz.size+1e-6); rd=np.sum(rz<80)/(rz.size+1e-6)
        if ld<min_rubber_dark*0.5 or rd<min_rubber_dark*0.5: continue
        separations.append({'x_center':int(px),'x_left':int(xl),'x_right':int(xr),
                             'width_px':int(xr-xl),'true_continuity':float(cont),'sep_type':'gap',
                             'mean_bright':col_bright,'left_dark':float(ld),'right_dark':float(rd)})

    # C: ESM overlap peaks
    esm_col = esm.mean(axis=0)
    esm_sm = cv2.GaussianBlur(esm_col.reshape(1,-1).astype(np.float32),(1,31),6).flatten()
    esm_peaks, _ = find_peaks(esm_sm, height=80, distance=30, prominence=30)
    for px in esm_peaks:
        if px<edge_px or px>W_r-edge_px: continue
        if any(abs(px-s['x_center'])<gap_merge_dist for s in separations): continue
        col_bright = float(gray_roi[:,max(0,px-8):min(W_r,px+9)].mean())
        if col_bright<overlap_bright_min or col_bright>bright_gap_thresh: continue
        esm_win = esm[:,max(0,px-5):min(W_r,px+6)].max(axis=1)
        cont = float(np.sum(esm_win>50)/H_r)
        if cont<min_vert_span: continue
        lz=gray_roi[:,max(0,px-context_pad):px]; rz=gray_roi[:,px:min(W_r,px+context_pad)]
        if lz.size==0 or rz.size==0: continue
        ld=np.sum(lz<80)/(lz.size+1e-6); rd=np.sum(rz<80)/(rz.size+1e-6)
        if ld<min_rubber_dark or rd<min_rubber_dark: continue
        separations.append({'x_center':int(px),'x_left':max(0,int(px-4)),'x_right':min(W_r,int(px+4)),
                             'width_px':8,'true_continuity':cont,'sep_type':'overlap',
                             'mean_bright':col_bright,'left_dark':float(ld),'right_dark':float(rd)})

    separations.sort(key=lambda s: s['x_center'])
    merged = []
    for s in separations:
        if (merged and abs(s['x_center']-merged[-1]['x_center'])<gap_merge_dist
                   and s['sep_type']!=merged[-1]['sep_type']):
            prev=merged[-1]
            merged[-1]={**prev,'x_left':min(prev['x_left'],s['x_left']),'x_right':max(prev['x_right'],s['x_right']),
                        'sep_type':'gap+overlap','true_continuity':max(prev['true_continuity'],s['true_continuity'])}
        else:
            merged.append(s)
    return merged


def detect_seps_vert_v2(gray_roi, esm, bright_thresh=110, min_rubber_dark=0.08,
                         context_pad=80, sep_half_height=25, min_peak_brightness=150, edge_margin=0.10):
    H_r, W_r = gray_roi.shape
    col_dark_per_row = np.sum(gray_roi<80, axis=1) / W_r
    rubber_rows = np.where(col_dark_per_row>0.20)[0]
    if len(rubber_rows)==0: return []
    rt, rb = int(rubber_rows.min()), int(rubber_rows.max())
    rh = rb - rt
    st = rt + int(rh*edge_margin); sb = rb - int(rh*edge_margin)
    if sb<=st: return []
    roi_r = gray_roi[st:sb,:]
    row_sm = cv2.GaussianBlur(roi_r.mean(axis=1).reshape(1,-1).astype(np.float32),(1,21),4).flatten()
    peaks, _ = find_peaks(row_sm, height=bright_thresh, distance=30, prominence=20)
    results = []
    for px in peaks:
        abs_py = st + px
        if float(row_sm[px]) < min_peak_brightness: continue
        top_z=gray_roi[max(0,abs_py-context_pad):abs_py,:]
        bot_z=gray_roi[abs_py:min(H_r,abs_py+context_pad),:]
        if top_z.size==0 or bot_z.size==0: continue
        td=np.sum(top_z<80)/(top_z.size+1e-6); bd=np.sum(bot_z<80)/(bot_z.size+1e-6)
        if td<min_rubber_dark or bd<min_rubber_dark: continue
        results.append({'y_center':int(abs_py),'y_top':max(0,int(abs_py-sep_half_height)),
                        'y_bot':min(H_r,int(abs_py+sep_half_height)),
                        'sep_type':'gap','mean_bright':float(row_sm[px])})
    return results


def run_pipeline(image_path):
    img=cv2.imread(image_path); gray=cv2.cvtColor(img,cv2.COLOR_BGR2GRAY); H_img,W_img=gray.shape
    strips=segment_rubber_strips(gray); pred=[]
    for strip in strips:
        x0,y0,w,h=strip['bbox']; roi=gray[y0:y0+h,x0:x0+w]
        esm,edm_x,edm_y=multiscale_esm(roi)
        if strip['orientation']=='horizontal':
            seps=detect_seps_horiz(roi,esm,edm_x,edm_y)
            for s in seps:
                bx=max(0,x0+s['x_left']-6); bx2=min(W_img,x0+s['x_right']+6)
                pred.append({'x':bx,'y':y0,'w':bx2-bx,'h':h,'cx':x0+s['x_center'],'cy':y0+h//2,
                             'orientation':'vertical','sep_type':s['sep_type']})
        else:
            seps=detect_seps_vert_v2(roi,esm)
            for s in seps:
                by=max(0,y0+s['y_top']-5); by2=min(H_img,y0+s['y_bot']+5)
                pred.append({'x':x0,'y':by,'w':w,'h':by2-by,'cx':x0+w//2,'cy':y0+s['y_center'],
                             'orientation':'horizontal','sep_type':s['sep_type']})
    return pred


print("Pipeline cargado.")

Pipeline cargado.


## Evaluación completa

In [4]:
def center_dist(b1, b2):
    return np.hypot(b1['cx']-b2['cx'], b1['cy']-b2['cy'])


def evaluate_dataset(gt, dataset_path, match_dist=MATCH_DIST_PX):
    tp = fp = fn = 0
    rows = []
    
    print(f"{'Image':<25} {'GT':>3} {'PRED':>5} {'TP':>3} {'FP':>3} {'FN':>3}  Status")
    print('-'*60)
    
    for fname, info in gt.items():
        path = os.path.join(dataset_path, fname)
        gt_bboxes = info['bboxes']
        
        try:
            pred_bboxes = run_pipeline(path)
        except Exception as e:
            pred_bboxes = []
            print(f"  ERROR {fname}: {e}")
        
        # Match by center distance
        matched_gt, matched_pred = set(), set()
        cands = [(center_dist(gb, pb), gi, pi)
                 for gi, gb in enumerate(gt_bboxes)
                 for pi, pb in enumerate(pred_bboxes)]
        for dist, gi, pi in sorted(cands):
            if gi not in matched_gt and pi not in matched_pred and dist < match_dist:
                matched_gt.add(gi); matched_pred.add(pi)
        
        tp_i = len(matched_gt)
        fp_i = len(pred_bboxes) - len(matched_pred)
        fn_i = len(gt_bboxes)  - len(matched_gt)
        tp += tp_i; fp += fp_i; fn += fn_i
        
        status = '✓' if fp_i==0 and fn_i==0 else '✗'
        rows.append({'fname':fname,'gt_n':len(gt_bboxes),'pred_n':len(pred_bboxes),
                     'tp':tp_i,'fp':fp_i,'fn':fn_i,'status':status,
                     'pred_bboxes':pred_bboxes,'gt_bboxes':gt_bboxes})
        print(f"{fname:<25} {len(gt_bboxes):>3} {len(pred_bboxes):>5} {tp_i:>3} {fp_i:>3} {fn_i:>3}  {status}")
    
    precision = tp/(tp+fp) if (tp+fp)>0 else 0
    recall    = tp/(tp+fn) if (tp+fn)>0 else 0
    f1        = 2*precision*recall/(precision+recall) if (precision+recall)>0 else 0
    perfect   = sum(1 for r in rows if r['status']=='✓')
    
    print(f"\n{'='*60}")
    print(f"TP={tp}  FP={fp}  FN={fn}")
    print(f"Precision: {precision:.1%}")
    print(f"Recall:    {recall:.1%}")
    print(f"F1:        {f1:.1%}")
    print(f"Imágenes perfectas: {perfect}/{len(rows)}")
    
    return rows, {'precision':precision,'recall':recall,'f1':f1,'tp':tp,'fp':fp,'fn':fn}


rows, metrics = evaluate_dataset(gt, DATASET_PATH)

Image                      GT  PRED  TP  FP  FN  Status
------------------------------------------------------------
  ERROR Multimedia__31_.jpg: OpenCV(4.13.0) D:\a\opencv-python\opencv-python\opencv\modules\imgproc\src\color.cpp:199: error: (-215:Assertion failed) !_src.empty() in function 'cv::cvtColor'

Multimedia__31_.jpg         2     0   0   0   2  ✗
  ERROR Multimedia__32_.jpg: OpenCV(4.13.0) D:\a\opencv-python\opencv-python\opencv\modules\imgproc\src\color.cpp:199: error: (-215:Assertion failed) !_src.empty() in function 'cv::cvtColor'

Multimedia__32_.jpg         2     0   0   0   2  ✗
  ERROR Multimedia__33_.jpg: OpenCV(4.13.0) D:\a\opencv-python\opencv-python\opencv\modules\imgproc\src\color.cpp:199: error: (-215:Assertion failed) !_src.empty() in function 'cv::cvtColor'

Multimedia__33_.jpg         2     0   0   0   2  ✗
  ERROR Multimedia__34_.jpg: OpenCV(4.13.0) D:\a\opencv-python\opencv-python\opencv\modules\imgproc\src\color.cpp:199: error: (-215:Assertion failed) !_sr

## Visualización de resultados

In [5]:
def visualize_result(row, dataset_path):
    """Muestra GT vs predicción para una imagen."""
    img = cv2.imread(os.path.join(dataset_path, row['fname']))
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    H, W = img_rgb.shape[:2]
    
    fig, axes = plt.subplots(1, 2, figsize=(16, 8))
    
    # GT
    gt_vis = img_rgb.copy()
    for b in row['gt_bboxes']:
        cv2.rectangle(gt_vis, (b['x'],b['y']), (b['x']+b['w'],b['y']+b['h']), (255,50,50), 4)
    axes[0].imshow(gt_vis)
    axes[0].set_title(f"Ground Truth ({row['gt_n']} seps)", fontsize=10)
    axes[0].axis('off')
    
    # Predicción
    pred_vis = img_rgb.copy()
    for b in row['pred_bboxes']:
        cv2.rectangle(pred_vis, (b['x'],b['y']), (b['x']+b['w'],b['y']+b['h']), (0,200,100), 4)
    axes[1].imshow(pred_vis)
    axes[1].set_title(
        f"Predicción ({row['pred_n']} seps) — "
        f"TP={row['tp']} FP={row['fp']} FN={row['fn']} {row['status']}",
        fontsize=10
    )
    axes[1].axis('off')
    
    plt.suptitle(row['fname'], fontsize=11, fontweight='bold')
    plt.tight_layout()
    plt.show()


# Mostrar imágenes con fallos
for row in rows:
    if row['status'] == '✗':
        visualize_result(row, DATASET_PATH)

error: OpenCV(4.13.0) D:\a\opencv-python\opencv-python\opencv\modules\imgproc\src\color.cpp:199: error: (-215:Assertion failed) !_src.empty() in function 'cv::cvtColor'
